# 04 — Entrenar M_seg (segmentación semántica foliar)

U-Net con encoder **ResNet50** (ImageNet transfer learning). Tres clases: `0=fondo`, `1=hoja_sana`, `2=hoja_enferma`. Severidad continua = `px_enferma / px_hoja × 100%`.

## Normalización cromática reproducible

Se usa **balance de blancos Gray-World** como única normalización (reproducible idéntica en Python y en el dispositivo Dart). La misma normalización se aplica en inferencia (backend y app), garantizando paridad train/serve para una severidad correcta.

## Pseudo-labels (HSV) + máscaras reales (COCO)

Las máscaras de las Fases 1–2 se generan automáticamente por análisis HSV (no anotaciones de experto): las métricas Dice/IoU miden similitud con la pseudo-label HSV. La Fase 3 incorpora ~700 máscaras reales anotadas (COCO Segmentation, hoja vs fondo) para detectar hojas en canopy denso y fondos complejos.

## Pipeline

1. Normalización Gray-World + pseudo-máscaras HSV.
2. **Fase 1**: U-Net ResNet50, decoder solo (encoder congelado), LR=1e-3.
3. **Fase 2**: fine-tune encoder (conv4+conv5) + Cosine LR.
4. **Fase 3** (~700 máscaras reales COCO): transfer learning (encoder congelado, decoder a LR bajo) + data augmentation fuerte + replay de muestras Fase 2 (anti-olvido), para robustez en fondos complejos sin degradar la textura sana/enferma.
5. Diagnóstico bias/varianza, visualización y export `model_seg.tflite` + `model_seg_int8.tflite`.

**Targets**: `val_dice_enferma ≥ 0.65`; mejorar IoU hoja vs fondo tras Fase 3.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib opencv-python-headless pycocotools

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = (256, 256)
NUM_CLASSES = 3
BATCH = 8
EPOCHS_P1 = 20
EPOCHS_P2 = 40
LR_P1 = 1e-3
LR_P2 = 1e-4
L2_DEFAULT = 2e-4

DATA = Path('./splits/clasificacion_patogeno')
OUT = Path('./outputs')
OUT.mkdir(exist_ok=True)

In [ ]:
import numpy as np
import cv2

_HSV_GREEN_LOW  = np.array([20, 25, 30])
_HSV_GREEN_HIGH = np.array([90, 255, 255])
_MORPH_KERNEL   = np.ones((5, 5), np.uint8)
_MORPH_LEAF     = np.ones((25, 25), np.uint8)


def chromatic_normalize(img_rgb: np.ndarray) -> np.ndarray:
    result = img_rgb.astype(np.float32)
    avg = result.reshape(-1, 3).mean(axis=0)
    scale = avg.mean() / (avg + 1e-6)
    return np.clip(result * scale, 0, 255).astype(np.uint8)


def _detect_leaf_region(hsv: np.ndarray) -> np.ndarray:
    _, s, v = cv2.split(hsv)
    _, mask_v = cv2.threshold(v, 28, 255, cv2.THRESH_BINARY)
    _, mask_s = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    leaf = cv2.bitwise_or(mask_v, mask_s)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_CLOSE, _MORPH_LEAF)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))

    contours, _ = cv2.findContours(leaf, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    refined = np.zeros_like(leaf)
    if contours:
        all_pts = np.vstack(contours)
        hull = cv2.convexHull(all_pts)
        cv2.fillPoly(refined, [hull], 255)
    else:
        refined = leaf
    return refined


def make_pseudo_mask(img_rgb: np.ndarray) -> np.ndarray:
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

    leaf_region = _detect_leaf_region(hsv)
    healthy = cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH)
    healthy = cv2.morphologyEx(healthy, cv2.MORPH_OPEN, _MORPH_KERNEL)

    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    mask[leaf_region > 0] = 2
    mask[healthy > 0] = 1
    return mask


def apply_leaf_mask(img_rgb: np.ndarray, mask: np.ndarray) -> np.ndarray:
    masked = img_rgb.copy()
    masked[mask == 0] = 0
    return masked


print('chromatic_normalize (gray-world), make_pseudo_mask y apply_leaf_mask definidos OK')


In [ ]:
import time
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence

DATA = globals().get('DATA', Path('./splits/clasificacion_patogeno'))
IMG_SIZE = globals().get('IMG_SIZE', (256, 256))
BATCH = globals().get('BATCH', 8)

_SEG_AUG = A.Compose([
    A.Rotate(limit=40, border_mode=0, p=0.6),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    A.ElasticTransform(alpha=60, sigma=6, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
])


def _safe_read(fp, size):
    for _ in range(3):
        try:
            return np.array(Image.open(fp).convert('RGB').resize(size))
        except (OSError, IOError):
            time.sleep(0.3)
    return None


class SegSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, augment, shuffle):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.samples = []
        for cls_dir in Path(directory).iterdir():
            if not cls_dir.is_dir():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                self.samples.extend(str(p) for p in cls_dir.glob(ext))
        self.n = len(self.samples)
        if shuffle:
            np.random.shuffle(self.samples)

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for fp in batch:
            img = _safe_read(fp, self.img_size)
            if img is None:
                continue
            img = chromatic_normalize(img)
            mask = make_pseudo_mask(img)
            if self.augment:
                aug = _SEG_AUG(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
            imgs.append(img.astype(np.float32) / 255.0)
            masks.append(mask[..., np.newaxis].astype(np.float32))
        if not imgs:
            return np.zeros((1, *self.img_size, 3), np.float32), np.zeros((1, *self.img_size, 1), np.float32)
        return np.stack(imgs), np.stack(masks)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)


train_gen = SegSequence(DATA / 'train', IMG_SIZE, BATCH, augment=True, shuffle=True)
val_gen   = SegSequence(DATA / 'val',   IMG_SIZE, BATCH, augment=False, shuffle=False)
print(f'Train: {train_gen.n} imgs | Val: {val_gen.n} imgs')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

_inspect_paths = train_gen.samples[:3]
fig, axes = plt.subplots(len(_inspect_paths), 2, figsize=(8, 4 * len(_inspect_paths)))
for row, fp in enumerate(_inspect_paths):
    raw = np.array(Image.open(fp).convert('RGB').resize(IMG_SIZE))
    norm = chromatic_normalize(raw)
    axes[row, 0].imshow(raw)
    axes[row, 0].set_title('Original'); axes[row, 0].axis('off')
    axes[row, 1].imshow(norm)
    axes[row, 1].set_title('Gray-World'); axes[row, 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

IMG_SIZE    = globals().get('IMG_SIZE', (256, 256))
NUM_CLASSES = globals().get('NUM_CLASSES', 3)
L2_DEFAULT  = globals().get('L2_DEFAULT', 2e-4)


def _decoder_block(x, skip, filters, l2):
    x = tf.keras.layers.UpSampling2D(2)(x)
    if skip is not None:
        x = tf.keras.layers.Concatenate()([x, skip])
    reg = tf.keras.regularizers.l2(l2)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.Activation('relu')(x)


def build_mseg(l2=L2_DEFAULT, dropout_rate=0.3):
    backbone = tf.keras.applications.ResNet50(
        input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    backbone.trainable = False

    s1 = backbone.get_layer('conv1_relu').output           # 128×128×64
    s2 = backbone.get_layer('conv2_block3_out').output     # 64×64×256
    s3 = backbone.get_layer('conv3_block4_out').output     # 32×32×512
    s4 = backbone.get_layer('conv4_block6_out').output     # 16×16×1024

    x = _decoder_block(backbone.output, s4, 512, l2)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = _decoder_block(x, s3, 256, l2)
    x = _decoder_block(x, s2, 128, l2)
    x = _decoder_block(x, s1, 64, l2)
    x = tf.keras.layers.UpSampling2D(2)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Conv2D(NUM_CLASSES, 1, activation='softmax')(x)

    return tf.keras.Model(inputs=backbone.input, outputs=x, name='M_seg_resnet50')


mseg = build_mseg()
mseg.summary(line_length=100)


In [ ]:
import tensorflow as tf

NUM_CLASSES = globals().get('NUM_CLASSES', 3)


def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_oh = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), -1), NUM_CLASSES)
    total = tf.constant(0.0)
    for c in range(NUM_CLASSES):
        p = y_pred[..., c]
        t = tf.cast(y_true_oh[..., c], tf.float32)
        inter = tf.reduce_sum(p * t)
        union = tf.reduce_sum(p) + tf.reduce_sum(t)
        total += 1.0 - (2 * inter + smooth) / (union + smooth)
    return total / tf.cast(NUM_CLASSES, tf.float32)


def seg_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(ce) + dice_loss(y_true, y_pred)


class DiceCoef(tf.keras.metrics.Metric):
    def __init__(self, class_id=0, name=None, **kwargs):
        super().__init__(name=name or f'dice_c{class_id}', **kwargs)
        self.class_id = class_id
        self.inter = self.add_weight(name='inter', shape=(), initializer='zeros')
        self.union = self.add_weight(name='union', shape=(), initializer='zeros')

    def get_config(self):
        base = super().get_config()
        base['class_id'] = self.class_id
        return base

    def update_state(self, y_true, y_pred, sample_weight=None):
        pred_c = tf.cast(tf.argmax(y_pred, -1) == self.class_id, tf.float32)
        true_c = tf.cast(tf.squeeze(y_true, -1) == self.class_id, tf.float32)
        self.inter.assign_add(tf.reduce_sum(pred_c * true_c))
        self.union.assign_add(tf.reduce_sum(pred_c) + tf.reduce_sum(true_c))

    def result(self):
        return (2 * self.inter + 1e-6) / (self.union + 1e-6)

    def reset_state(self):
        self.inter.assign(0.0)
        self.union.assign(0.0)


def compile_mseg(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=seg_loss,
        metrics=[
            DiceCoef(0, 'dice_bg'),
            DiceCoef(1, 'dice_sana'),
            DiceCoef(2, 'dice_enferma'),
        ],
    )


print('seg_loss, DiceCoef y compile_mseg definidos OK')

In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P1 = globals().get('EPOCHS_P1', 30)
LR_P1     = globals().get('LR_P1', 1e-3)
OUT       = globals().get('OUT', Path('./outputs'))

for layer in mseg.layers:
    layer.trainable = False
for layer in mseg.layers:
    if any(layer.name.startswith(p) for p in ('up_sampling', 'conv2d', 'concatenate', 'dropout', 'batch_normalization', 'activation')):
        layer.trainable = True

compile_mseg(mseg, LR_P1)

cbs_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_enferma', mode='max', patience=8,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / 'model_seg_p1_best.keras'),
        monitor='val_dice_enferma', mode='max', save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1,
    ),
]

print('=== FASE 1: decoder (ResNet50 encoder congelado) ===')
h1 = mseg.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=cbs_p1, verbose=1)


In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P2 = globals().get('EPOCHS_P2', 40)
LR_P2     = globals().get('LR_P2', 1e-4)
BATCH     = globals().get('BATCH', 8)
OUT       = globals().get('OUT', Path('./outputs'))

for layer in mseg.layers:
    if any(s in layer.name for s in ('conv4_block', 'conv5_block')):
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True

steps = max(1, train_gen.n // BATCH) * EPOCHS_P2
lr_sched = tf.keras.optimizers.schedules.CosineDecay(LR_P2, steps, alpha=0.05)
compile_mseg(mseg, lr_sched)

initial_epoch = len(h1.history['loss'])

cbs_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_enferma', mode='max', patience=10,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / 'model_seg_best.keras'),
        monitor='val_dice_enferma', mode='max', save_best_only=True, verbose=1,
    ),
]

print('=== FASE 2: fine-tune ResNet50 conv4+conv5 + Cosine LR ===')
h2 = mseg.fit(
    train_gen, validation_data=val_gen,
    epochs=initial_epoch + EPOCHS_P2, initial_epoch=initial_epoch,
    callbacks=cbs_p2, verbose=1,
)
mseg.save(OUT / 'model_seg.keras')
print('Modelo guardado en', OUT / 'model_seg.keras')


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUT = globals().get('OUT', Path('./outputs'))


def plot_training(h1, h2, title='M_seg'):
    loss_all  = h1.history['loss'] + h2.history['loss']
    vloss_all = h1.history['val_loss'] + h2.history['val_loss']
    dice_s    = h1.history.get('dice_sana', []) + h2.history.get('dice_sana', [])
    vdice_s   = h1.history.get('val_dice_sana', []) + h2.history.get('val_dice_sana', [])
    dice_e    = h1.history.get('dice_enferma', []) + h2.history.get('dice_enferma', [])
    vdice_e   = h1.history.get('val_dice_enferma', []) + h2.history.get('val_dice_enferma', [])

    ep  = range(1, len(loss_all) + 1)
    div = len(h1.history['loss'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(ep, loss_all, 'b-', label='train_loss')
    axes[0].plot(ep, vloss_all, 'r-', label='val_loss')
    axes[0].axvline(div, color='gray', linestyle='--', label='fine-tune')
    axes[0].set_title(f'{title} — Loss (Dice + CE)')
    axes[0].legend(); axes[0].grid(True)

    if dice_s:
        axes[1].plot(ep, dice_s,  'g-',  label='train_dice_sana')
        axes[1].plot(ep, vdice_s, 'g--', label='val_dice_sana')
    if dice_e:
        axes[1].plot(ep, dice_e,  'r-',  label='train_dice_enferma')
        axes[1].plot(ep, vdice_e, 'r--', label='val_dice_enferma')
    axes[1].axvline(div, color='gray', linestyle='--')
    axes[1].set_title(f'{title} — Dice por clase')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(OUT / f'{title}_curves.png', dpi=120)
    plt.show()


def diagnose_bias_variance(h1, h2, model_name='M_seg'):
    train_loss = (h1.history['loss'] + h2.history['loss'])[-1]
    val_loss   = (h1.history['val_loss'] + h2.history['val_loss'])[-1]
    gap = val_loss - train_loss
    print(f'[{model_name}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | gap={gap:.4f}')
    if gap > 0.15:
        print('  ALTA VARIANZA — overfitting. Aumentar dropout, L2, data aug, early stopping.')
    elif train_loss > 0.3:
        print('  ALTO BIAS — underfitting. Aumentar capacidad, mas epocas, menos regularizacion.')
    else:
        print('  Bias/varianza balanceados.')
    dice_e_val = (h1.history.get('val_dice_enferma', [0]) + h2.history.get('val_dice_enferma', [0]))[-1]
    print(f'  val_dice_enferma final: {dice_e_val:.4f}  (objetivo >= 0.60) {"OK" if dice_e_val >= 0.60 else "REVISAR"}')


plot_training(h1, h2, title='M_seg')
diagnose_bias_variance(h1, h2, 'M_seg')


In [ ]:
from pathlib import Path
import tensorflow as tf

DATA      = globals().get('DATA', Path('./splits/clasificacion_patogeno'))
OUT       = globals().get('OUT', Path('./outputs'))
IMG_SIZE  = globals().get('IMG_SIZE', (256, 256))
BATCH     = globals().get('BATCH', 8)
EPOCHS_P1 = globals().get('EPOCHS_P1', 20)
LR_P1     = globals().get('LR_P1', 1e-3)

print('=== Sweep de lambda (L2) ===')
sweep_results = []
for l2_val in [0.0, 1e-5, 1e-4, 1e-3]:
    m = build_mseg(l2=l2_val)
    compile_mseg(m, LR_P1)
    h = m.fit(
        SegSequence(DATA / 'train', IMG_SIZE, BATCH, augment=True, shuffle=True),
        validation_data=SegSequence(DATA / 'val', IMG_SIZE, BATCH, augment=False, shuffle=False),
        epochs=15, verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        ],
    )
    tl = h.history['loss'][-1]
    vl = h.history['val_loss'][-1]
    sweep_results.append((l2_val, tl, vl, vl - tl))
    print(f'  lambda={l2_val:.0e} -> train={tl:.4f} val={vl:.4f} gap={vl-tl:.4f}')

best = min(sweep_results, key=lambda r: r[2])
print(f'\nMejor lambda: {best[0]:.0e} (val_loss={best[2]:.4f})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_COLORS = np.array([[0,0,0], [34,197,94], [249,115,22]], dtype=np.uint8)


def visualize_predictions(generator, model, n=5):
    x_batch, y_batch = generator[0]
    preds = model.predict(x_batch[:n], verbose=0)
    pred_masks = np.argmax(preds, axis=-1)
    true_masks = y_batch[:n, ..., 0].astype(int)

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3))
    for i in range(min(n, len(x_batch))):
        img = (x_batch[i] * 255).astype(np.uint8)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Original'); axes[i, 0].axis('off')
        axes[i, 1].imshow(_COLORS[true_masks[i]])
        axes[i, 1].set_title('Mascara HSV'); axes[i, 1].axis('off')
        axes[i, 2].imshow(_COLORS[pred_masks[i]])
        axes[i, 2].set_title('Pred M_seg'); axes[i, 2].axis('off')
    plt.tight_layout()
    plt.show()


visualize_predictions(val_gen, mseg, n=5)

## Fase 3 — máscaras reales COCO (fondos complejos)

Ajuste fino del modelo de Fase 2 con ~700 máscaras anotadas (COCO Segmentation, clase hoja vs fondo) en `splits/masks/`, dirigidas a **canopy denso y fondos complejos** (suelo, varias hojas). Encoder congelado, decoder a LR muy bajo, augmentación fuerte y *replay* de muestras pseudo-HSV de Fase 2 para no olvidar la textura sana/enferma. Corrige la confusión hoja/suelo. Requiere `pycocotools`; si no existe el archivo de anotaciones, la fase se omite.

In [ ]:
from pathlib import Path
import numpy as np
import cv2
import tensorflow as tf
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence

MASKS_DIR = Path('./splits/masks')
COCO_ANN = MASKS_DIR / '_annotations.coco.json'
IMG_SIZE = globals().get('IMG_SIZE', (256, 256))
BATCH = globals().get('BATCH', 8)
OUT = globals().get('OUT', Path('./outputs'))
EPOCHS_P3 = 10
LR_P3 = 1e-5
REPLAY_RATIO = 0.3

_P3_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=0.6),
    A.RandomShadow(p=0.3),
    A.Affine(translate_percent=0.1, scale=(0.85, 1.2), rotate=0, p=0.5),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
])


def _coco_to_3class(img_rgb, leaf_bin):
    norm = chromatic_normalize(img_rgb)
    hsv = cv2.cvtColor(cv2.cvtColor(norm, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    healthy = cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH)
    mask = np.zeros(leaf_bin.shape, np.uint8)
    mask[leaf_bin > 0] = 2
    mask[(leaf_bin > 0) & (healthy > 0)] = 1
    return norm, mask


class Phase3Sequence(Sequence):
    def __init__(self, coco, img_ids, img_dir, replay_samples, img_size, batch_size, augment):
        self.coco = coco
        self.img_ids = img_ids
        self.img_dir = Path(img_dir)
        self.replay = replay_samples
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment

    def __len__(self):
        return max(1, len(self.img_ids) // self.batch_size)

    def _load_coco(self, img_id):
        info = self.coco.loadImgs(img_id)[0]
        raw = np.array(Image.open(self.img_dir / info['file_name']).convert('RGB'))
        leaf = np.zeros((info['height'], info['width']), np.uint8)
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            leaf = np.maximum(leaf, self.coco.annToMask(ann))
        raw = cv2.resize(raw, self.img_size)
        leaf = cv2.resize(leaf, self.img_size, interpolation=cv2.INTER_NEAREST)
        return _coco_to_3class(raw, leaf)

    def _load_replay(self, fp):
        raw = np.array(Image.open(fp).convert('RGB').resize(self.img_size))
        norm = chromatic_normalize(raw)
        return norm, make_pseudo_mask(norm)

    def __getitem__(self, idx):
        batch_ids = self.img_ids[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for img_id in batch_ids:
            if self.replay and np.random.rand() < REPLAY_RATIO:
                img, mask = self._load_replay(np.random.choice(self.replay))
            else:
                img, mask = self._load_coco(img_id)
            if self.augment:
                aug = _P3_AUG(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
            imgs.append(img.astype(np.float32) / 255.0)
            masks.append(mask[..., np.newaxis].astype(np.float32))
        if not imgs:
            return np.zeros((1, *self.img_size, 3), np.float32), np.zeros((1, *self.img_size, 1), np.float32)
        return np.stack(imgs), np.stack(masks)


if not COCO_ANN.exists():
    print('Fase 3 omitida: no se encontro', COCO_ANN)
else:
    from pycocotools.coco import COCO

    coco = COCO(str(COCO_ANN))
    all_ids = coco.getImgIds()
    np.random.shuffle(all_ids)
    split = int(len(all_ids) * 0.85)
    train_ids, val_ids = all_ids[:split], all_ids[split:]
    replay = train_gen.samples

    p3_train = Phase3Sequence(coco, train_ids, MASKS_DIR, replay, IMG_SIZE, BATCH, augment=True)
    p3_val = Phase3Sequence(coco, val_ids, MASKS_DIR, [], IMG_SIZE, BATCH, augment=False)

    for layer in mseg.layers:
        layer.trainable = not any(s in layer.name for s in ('conv1', 'conv2_block', 'conv3_block', 'conv4_block', 'conv5_block'))
    compile_mseg(mseg, LR_P3)

    print(f'=== FASE 3: {len(train_ids)} train / {len(val_ids)} val (mascaras COCO) + replay ===')
    cbs_p3 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_dice_enferma', mode='max', patience=4,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(OUT / 'model_seg_best.keras'),
            monitor='val_dice_enferma', mode='max', save_best_only=True, verbose=1,
        ),
    ]
    h3 = mseg.fit(p3_train, validation_data=p3_val, epochs=EPOCHS_P3, callbacks=cbs_p3, verbose=1)
    mseg.save(OUT / 'model_seg.keras')
    print('Fase 3 completada. Modelo guardado en', OUT / 'model_seg.keras')


In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

OUT = globals().get('OUT', Path('./outputs'))

converter = tf.lite.TFLiteConverter.from_keras_model(mseg)
tflite_float = converter.convert()
Path(OUT / 'model_seg.tflite').write_bytes(tflite_float)
size_f32 = Path(OUT / 'model_seg.tflite').stat().st_size / (1024 * 1024)
print(f'model_seg.tflite: {size_f32:.2f} MB (float32)')


def _rep_dataset():
    for i in range(min(50, len(val_gen))):
        x, _ = val_gen[i]
        for img in x:
            yield [img[np.newaxis].astype(np.float32)]


converter_int8 = tf.lite.TFLiteConverter.from_keras_model(mseg)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = _rep_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()
Path(OUT / 'model_seg_int8.tflite').write_bytes(tflite_int8)
size_int8 = Path(OUT / 'model_seg_int8.tflite').stat().st_size / (1024 * 1024)
print(f'model_seg_int8.tflite: {size_int8:.2f} MB (int8)')

print('\nCopiar a App/assets/models/seg/model_seg.tflite  (preferir int8 si < 4 MB)')